# 02 · Build the Qdrant index

Run from the project folder after `pip install -r requirements.txt`. Everything is written inside the project:

* `data/stories.parquet`: local copy of `FareedKhan/1k_stories_100_genre`, so later steps never need the Hub
* `data/splits.json`: the frozen 7/1/2 split shared with Task 2 (if it already exists, it is reused)
* `qdrant_data/`: the Qdrant database (local on-disk mode) with two collections
  * `story_chunks`: named vectors `dense_late` (late chunking) and `dense_prefix` (baseline, kept for the A/B), sparse `sparse` (BM25), payload `story_id`, `title`, `genre`, `chunk_index`, `text`
  * `stories`: full story text by id, for parent-document retrieval and id/title lookups
* `qdrant_data/build_manifest.json`: counts, timings, settings. Its presence means the build finished.

Re-running this notebook does **not** rebuild or wipe the index. Set `REBUILD = True` to force it.

All logic lives in `src/storybot/`; settings (including the database folder) live in `configs/retrieval.yaml`.

## 1. Settings

In [ ]:
import torch
from storybot.config import load_config, resolve

cfg = load_config()          # configs/retrieval.yaml
REBUILD = False
LIMIT = None                 # e.g. 50 for a quick test run

print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("embedder:", cfg.embedding.model)
print("Qdrant database folder:", resolve(cfg, cfg.qdrant.path))

## 2. Data: local copy and frozen split

In [ ]:
from storybot.data import prepare_data

df = prepare_data(cfg, limit=LIMIT)
df.head()

## 3. Build the collections

Steps (see `storybot/retrieval/indexing.py`):
1. Sentence-aware chunks of ~400 tokens with ~50 tokens overlap; stories up to 1,000 tokens stay whole.
2. **Late chunking**: `search_document: ` + title/genre line + whole story go through ModernBERT once (8k context); token vectors are mean-pooled per chunk span.
3. **Prefix baseline**: each chunk embedded alone with the same prefix (`max_length` 1,100, so whole-story chunks are not cut at 512).
4. **BM25** on title/genre line + chunk text, without the `search_document: ` prefix.
5. Upload to Qdrant, then write the manifest.

In [ ]:
from storybot.retrieval import build_index

manifest = build_index(cfg, df, rebuild=REBUILD)

## 4. Check the index

In [ ]:
from qdrant_client import models as m
from storybot.retrieval import open_client, DENSE_LATE, SPARSE
from storybot.retrieval.embed import DenseEmbedder, SparseEncoder

client = open_client(cfg)    # same client as the build; local mode allows one per folder
for name in (cfg.qdrant.chunks_collection, cfg.qdrant.stories_collection):
    print(name, client.count(name, exact=True).count)

e = cfg.embedding
dense = DenseEmbedder(e.model, fp16=e.fp16, doc_prefix=e.doc_prefix, query_prefix=e.query_prefix)
bm25 = SparseEncoder(cfg.sparse.model)

def hybrid(query, genre=None, limit=5):
    flt = m.Filter(must=[m.FieldCondition(key="genre", match=m.MatchValue(value=genre))]) if genre else None
    qi, qv = bm25.embed_query(query)
    return client.query_points(cfg.qdrant.chunks_collection, prefetch=[
        m.Prefetch(query=dense.embed_query(query).tolist(), using=DENSE_LATE, limit=30, filter=flt),
        m.Prefetch(query=m.SparseVector(indices=qi, values=qv), using=SPARSE, limit=30, filter=flt)],
        query=m.FusionQuery(fusion=m.Fusion.RRF), limit=limit, with_payload=True).points

q = "a detective investigating strange disappearances in a quiet town"
for h in hybrid(q):
    print(round(h.score, 3), h.payload["story_id"], h.payload["title"], "| chunk", h.payload["chunk_index"])
print("\nfiltered to Mystery:")
for h in hybrid(q, genre="Mystery")[:3]:
    print(round(h.score, 3), h.payload["title"], h.payload["genre"])